# Affordable Housing Stability & Eviction Prevention—Load & Explore

**Agents for Impact 2026 · Affordable Housing Stability (`A4I2026-challenge-housing`)**

You are building an agent for an **Eviction Prevention & Emergency Rental Assistance Triage Coordinator**—internal municipal and legal-aid intake staff who must allocate scarce emergency rental relief dollars, habitability repair escrows, and right-to-counsel slots before families lose their homes. Your tool is **not** a tenant-facing chatbot, **not** legal advice, and **not** an automated eligibility denial engine: a human coordinator reviews every case.

This notebook loads, cleans, and validates your five-table BigQuery foundation. It stops right before building the **BigQuery Vector Search** (`AI.EMBED` + `VECTOR_SEARCH`) pipeline and your deployed agent—that part is yours.

## What you'll have when this finishes

| Table | Grain | Source & Provenance | What it tells you |
|---|---|---|---|
| `acs_tract_housing_tenure` | 1 row / census tract | **Real public data** (US Census ACS 5-Year `B25070` / `B19013`) | Renter households, moderate (`30–49%`) and severe (`50%+`) rent burden, median rent, poverty, vehicle access, and post-output `audit_*` shares |
| `court_eviction_filings` | 1 row / tract × quarter | **`[Modeled]`** from county civil court baselines & ACS pressure (`2025-Q3`..`2026-Q2`) | Quarterly nonpayment filings, holdover/no-fault filings, default judgments, and median claimed arrears |
| `municipal_311_housing_complaints` | 1 row / 311 case | **Real municipal 311 open data** (`165`/metro) + **`9 [Planted Case]` decoys**/metro | Block-coarsened habitability complaints, inspection status, open/closed dates, and unstructured citizen/inspector narratives |
| `rental_relief_program_bulletins` | 1 row / relief program | **`[Curated Benchmark]`** (`4` programs/metro) | Municipal and nonprofit arrears grants, AMI caps, payment ceilings, and whether an open habitability case is required |
| `municipal_housing_ordinance_statutes` | 1 row / statute | **`[Public Legal Reference]`** (`3` statutes/metro) | State and local implied warranty of habitability repair deadlines, rent-escrow rules, and anti-retaliation windows |

## The four lanes, and how to spend your 4.5 hours

- **Data lane.** Run this notebook (~2 minutes), verify all Section 12 checks pass, then build the semantic `AI.EMBED` + `VECTOR_SEARCH` layer from Section 13 and your compounding risk joins.
- **Agent lane.** Design an internal coordinator triage workflow that combines structured tract displacement pressure, semantic 311 habitability evidence, and statutory/program eligibility rules—while citing every claim's provenance.
- **Front-end lane.** Build a triage console for intake coordinators against the five BigQuery tables immediately; wire it to your live Cloud Run / Agent Runtime endpoint once deployed.
- **Story lane.** Your submission is graded asynchronously from your **GitHub repo**, **Google Skills project**, **pitch deck**, and **3-minute screen-share video**. Lead with a surprising empirical finding from your metro and state plainly what is real vs. modeled or planted.

## Before you start

Two prompts appear the first time you open **Colab Enterprise** in a fresh Google Cloud project, and both are expected:

1. Opening Colab Enterprise prompts you to **enable required APIs**—accept them.
2. The Colab Enterprise header may display a second **"Enable APIs"** banner—enable that too.

Everything this notebook imports (`google-cloud-bigquery`, `pandas`, `pyarrow`, `requests`) is already pre-installed on the standard Colab Enterprise runtime. Do not `pip install` extra packages here.

**How to run:** Click the next code cell, choose your `METRO` (or keep `"San Francisco"`), and select **Runtime → Run all**. Every query runs through the Python `bigquery.Client`—no `%%bigquery` cell magics—so execution is deterministic top to bottom.

## 1. Setup

Change `METRO` below if your team wants a different city. Section 3 compares all six pre-staged metros (`San Francisco`, `Chicago`, `New York`, `Austin`, `Baton Rouge`, and `Cincinnati`) with exact measured counts so you know what each city looks like before committing.

In [ ]:
# ---- the one thing you change ------------------------------------------------
METRO    = "San Francisco"   # Choose from METROS below (default: San Francisco, CA)
DATASET  = "a4i_housing"     # BigQuery dataset created in your own GCP project
LOCATION = "US"              # Keep in US so Vertex AI connection us.a4i-vertex co-locates
# ------------------------------------------------------------------------------

import contextlib
import io
import json
import os
from pathlib import Path
import re
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from google.cloud import bigquery

METROS = {
    "San Francisco": ("CA", "06", "san-francisco"),
    "Chicago":       ("IL", "17", "chicago"),
    "New York":      ("NY", "36", "new-york"),
    "Austin":        ("TX", "48", "austin"),
    "Baton Rouge":   ("LA", "22", "baton-rouge"),
    "Cincinnati":    ("OH", "39", "cincinnati"),
}
if METRO not in METROS:
    raise ValueError(f"Unknown METRO={METRO!r}. Choose one of: {list(METROS)}")

STATE_ABBR, STATE_FIPS, METRO_SLUG = METROS[METRO]

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT:
    PROJECT = bigquery.Client().project
if not PROJECT:
    raise RuntimeError("Could not determine GCP project. Set GOOGLE_CLOUD_PROJECT.")

bq = bigquery.Client(project=PROJECT, location=LOCATION)
ds_ref = bigquery.Dataset(f"{PROJECT}.{DATASET}")
ds_ref.location = LOCATION
bq.create_dataset(ds_ref, exists_ok=True)

print(f"project     : {PROJECT}")
print(f"metro       : {METRO}, {STATE_ABBR} (state FIPS {STATE_FIPS}, slug {METRO_SLUG})")
print(f"dataset     : {PROJECT}.{DATASET} ({LOCATION})")
print(f"pandas      : {pd.__version__}")

### Timing, two-tier validation, and resilient GeoParquet loading

Every stage below runs inside `step(...)` so slow calls surface immediately, and every data check records into `CHECKS` under one of two verdicts:

- **`check()` (`FAIL` when false)** — grades **our pipeline**: key lengths, FIPS prefixes, row-count floors, foreign keys, and PII redaction. A `FAIL` means downstream outputs cannot be trusted.
- **`note()` (`WARN` up to `limit`, `FAIL` above it)** — surfaces **upstream public-data quirks** without halting a valid load unless systemic.

When students import this notebook into Colab Enterprise by raw GitHub URL, the runtime has no local `../data/` directory. `read_starter_parquet()` checks local paths first and falls back cleanly over HTTPS with bounded retries and Parquet magic-byte verification (`PAR1`). `load_table()` preserves GeoParquet v1.0.0 spherical WKB metadata so `tract_geom` and `property_geom` land in BigQuery as native `GEOGRAPHY` columns.

In [ ]:
STEPS, CHECKS = {}, []
NOTEBOOK_START = time.time()

GITHUB_ORG = os.environ.get("A4I_GITHUB_ORG", "Bornism")
REPO_NAME = "A4I2026-challenge-housing"
SNAPSHOT_BUCKET = os.environ.get("A4I_SNAPSHOT_BUCKET", "class-demo")
SNAPSHOT_PREFIX = "a4i-2026/challenge-housing"
UA = {"User-Agent": "A4I2026-challenge-housing-notebook"}


@contextlib.contextmanager
def step(name):
    t0 = time.time()
    print(f"[{name}] ...")
    try:
        yield
    finally:
        dt = time.time() - t0
        STEPS[name] = round(dt, 1)
        print(f"[{name}] {dt:.1f}s")


def _record(name, verdict, detail):
    for c in CHECKS:
        if c["check"] == name:
            c["verdict"], c["detail"] = verdict, detail
            return
    CHECKS.append({"check": name, "verdict": verdict, "detail": detail})


def check(name, ok, detail=""):
    _record(name, "PASS" if ok else "FAIL", detail)
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f": {detail}" if detail else ""))
    return bool(ok)


def note(name, bad, total, detail="", limit=0.01, rows=None):
    frac = (bad / total) if total else 0.0
    verdict = "PASS" if bad == 0 else ("FAIL" if (total and frac > limit) else "WARN")
    msg = detail or (f"{bad:,} of {total:,}" if total else f"{bad:,}")
    _record(name, verdict, msg)
    print(f"  {verdict}  {name}: {msg}")
    if verdict != "PASS" and rows is not None:
        display(rows.head(10))
    return verdict


def safe_str(s, width=None):
    out = s.astype("string")
    if width:
        out = out.str.strip().str.zfill(width)
    return out.where(out.notna(), None)


def spread(series, label=""):
    s = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if s.empty:
        return {}
    p10, p50, p90 = s.quantile([0.10, 0.50, 0.90])
    out = {
        "n": int(s.size),
        "min": round(float(s.min()), 3),
        "p10": round(float(p10), 3),
        "median": round(float(p50), 3),
        "p90": round(float(p90), 3),
        "max": round(float(s.max()), 3),
    }
    if label:
        print(f"  {label:<30} " + "  ".join(f"{k}={v}" for k, v in out.items()))
    return out

In [ ]:
def _fetch_parquet_bytes(url, tries=4, timeout=60):
    delay = 1.5
    for attempt in range(tries):
        try:
            resp = requests.get(url, timeout=timeout, headers=UA)
        except requests.RequestException as exc:
            if attempt == tries - 1:
                raise RuntimeError(f"{url} — {type(exc).__name__}: {exc}") from exc
            time.sleep(delay)
            delay *= 2
            continue
        if resp.status_code == 429 or 500 <= resp.status_code < 600:
            if attempt == tries - 1:
                raise RuntimeError(f"{url} — HTTP {resp.status_code} after {tries} tries")
            time.sleep(delay)
            delay *= 2
            continue
        if resp.status_code != 200:
            raise RuntimeError(f"{url} — HTTP {resp.status_code}")
        if resp.content[:4] != b"PAR1":
            raise RuntimeError(f"{url} — response is not a valid Parquet file")
        return resp.content
    raise RuntimeError(f"{url} — exhausted retries")


def _fetch_gcs_parquet_bytes(bucket_name, blob_path):
    from google.cloud import storage as gcs_storage
    client = gcs_storage.Client(project=PROJECT)
    blob = client.bucket(bucket_name).blob(blob_path)
    data = blob.download_as_bytes(timeout=60)
    if data[:4] != b"PAR1":
        raise RuntimeError(f"gs://{bucket_name}/{blob_path} is not a valid Parquet file")
    return data


def read_starter_parquet(filename):
    if not filename or "/" in filename or ".." in filename:
        raise ValueError(f"Invalid starter filename: {filename!r}")
    data_dir = os.environ.get("A4I_DATA_DIR", "").strip()
    local_candidates = (
        ([Path(data_dir) / filename] if data_dir else [])
        + [Path("../data") / filename, Path("data") / filename, Path(filename)]
    )
    for path in local_candidates:
        if path.is_file():
            return pd.read_parquet(path)

    org_candidates = list(dict.fromkeys([GITHUB_ORG, "Bornism", "haggman"]))
    base_override = os.environ.get("A4I_DATA_BASE_URL", "").strip().rstrip("/")
    urls = ([f"{base_override}/{filename}"] if base_override else []) + [
        f"https://raw.githubusercontent.com/{org}/{REPO_NAME}/main/data/{filename}"
        for org in org_candidates
    ]
    errors = []
    for url in urls:
        try:
            return pd.read_parquet(io.BytesIO(_fetch_parquet_bytes(url)))
        except RuntimeError as exc:
            errors.append(str(exc))

    if SNAPSHOT_BUCKET:
        blob_path = f"{SNAPSHOT_PREFIX}/starter/{filename}"
        try:
            return pd.read_parquet(io.BytesIO(_fetch_gcs_parquet_bytes(SNAPSHOT_BUCKET, blob_path)))
        except Exception as exc:  # noqa: BLE001
            errors.append(f"gs://{SNAPSHOT_BUCKET}/{blob_path} ({type(exc).__name__}: {exc})")

    raise RuntimeError(f"Could not locate {filename} locally or remotely: {'; '.join(errors)}")

def _to_geoparquet_buffer(df, geom_col=None):
    table = pa.Table.from_pandas(df, preserve_index=False)
    if geom_col and geom_col in df.columns:
        geo_meta = {
            "version": "1.0.0",
            "primary_column": geom_col,
            "columns": {
                geom_col: {
                    "encoding": "WKB",
                    "geometry_types": [],
                    "crs": None,
                    "orientation": "counterclockwise",
                    "edges": "spherical",
                }
            },
        }
        meta = dict(table.schema.metadata or {})
        meta[b"geo"] = json.dumps(geo_meta).encode("utf-8")
        table = table.replace_schema_metadata(meta)
    buf = io.BytesIO()
    pq.write_table(table, buf)
    buf.seek(0)
    return buf


def load_table(df, table_name, geom_col=None, description=""):
    if df is None or not table_name:
        raise ValueError("df and table_name are required")
    ref = f"{PROJECT}.{DATASET}.{table_name}"
    buf = _to_geoparquet_buffer(df, geom_col=geom_col)
    cfg = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    bq.load_table_from_file(buf, ref, job_config=cfg).result()
    tbl = bq.get_table(ref)
    if description:
        tbl.description = description
        bq.update_table(tbl, ["description"])
    print(f"  loaded {table_name:<38} {tbl.num_rows:>6,} rows")
    return tbl

## 2. Start with the wrong answer

Two of the four emergency rental relief programs in your metro (`requires_open_habitability_case = TRUE`) require **active residential habitability violations**—conditions like heat loss, severe mold/water intrusion, raw sewage backups, or electrical arcing.

The obvious shortcut is a SQL keyword filter (`LIKE '%no heat%' OR LIKE '%mold%'`). Let's run that exact filter on your metro's 311 table right now and watch it fail in **both** directions at once:

1. **False negatives (missed emergencies):** Citizens and housing inspectors rarely type the exact phrase `"no heat"`. They write *"radiators stone cold"*, *"boiler shutdown"*, *"water intrusion onto breaker panel"*, or *"plumbing overflowing"*.
2. **False positives (`[Planted Case]` decoys):** Keyword search happily matches a **commercial warehouse** (`"no heat ... zero residential tenants"`), a **closed/abated repair** (`"black mold ... remediation completed"`), and a **proactive clearance** whose inspector literally wrote `"no mold, no leaks, and zero habitability defects found"`!

In [ ]:
with step("section 2 wrong-answer keyword hook"):
    _raw_311 = read_starter_parquet("municipal_311_housing_complaints.parquet")
    _m311 = _raw_311[_raw_311["metro_area"] == METRO].copy()

    kw_mask = _m311["complaint_narrative"].str.contains(
        r"no heat|mold", case=False, na=False, regex=True
    )
    kw_hits = _m311[kw_mask].copy()

    decoy_hits = kw_hits[
        (kw_hits["property_type"] != "Residential") | (kw_hits["closed_date"].notna())
    ]
    missed_thermal_moisture = _m311[
        (_m311["property_type"] == "Residential")
        & (_m311["closed_date"].isna())
        & (~kw_mask)
        & _m311["complaint_narrative"].str.contains(
            r"boiler|radiator|hot water|sewage|sewer|water intrusion|leak|electrical|rodent",
            case=False,
            na=False,
            regex=True,
        )
    ]

    print(f"Total 311 complaints in {METRO}: {len(_m311):,}")
    print(f"Naive 'no heat | mold' keyword matches : {len(kw_hits):,}")
    print(f"  of which are INELIGIBLE (Commercial or Closed): {len(decoy_hits):,}")
    print(f"Open residential hazard cases MISSED by keyword : {len(missed_thermal_moisture):,}")
    print("\nIneligible rows pulled in by naive keyword matching:")
    display(
        decoy_hits[
            ["request_id", "property_type", "inspection_status", "closed_date", "complaint_narrative"]
        ].head(6)
    )

That is why **BigQuery Vector Search** (`AI.EMBED` + `VECTOR_SEARCH`) is your technical differentiator—and why Vector Search **alone** is still not enough unless you pair semantic similarity with hard post-search relational predicates (`property_type = 'Residential' AND closed_date IS NULL`). Every metro carries **9 `[Planted Case]` decoys** (`DECOY-*`) specifically designed to test whether your pipeline enforces both.

## 3. Choosing a metro

All six metros below are pre-staged and verified. Notice two structural differences before you choose:

- **Leading-zero FIPS (`San Francisco`):** California's state FIPS is `'06'`. Cast `census_tract_geoid` to an integer anywhere in Python or SQL and `'06075010800'` becomes `6075010800` (10 digits), silently breaking every tract join.
- **Open-case mix (`Baton Rouge` vs. `New York` / `Chicago`):** Baton Rouge has the highest severe rent burden (`27.2%`) in the pack, but `155` of its `165` real 311 requests are already closed—leaving only `10` open residential 311 cases. That is a real municipal intake dynamic, not a broken load.

| Metro | State (`FIPS`) | Tracts | Renter units | Severe rent burden (`50%+`) | Annual modeled filings | Open residential 311s | `[Planted Case]` decoys |
|---|---|---:|---:|---:|---:|---:|---:|
| **`San Francisco`** *(default)* | `CA` (`06`) | 42 | 66,252 | 17.5% | 2,359 | 140 | 9 |
| `Chicago` | `IL` (`17`) | 42 | 100,465 | 18.9% | 6,563 | 165 | 9 |
| `New York` | `NY` (`36`) | 168 | 437,111 | 25.2% | 45,471 | 165 | 9 |
| `Austin` | `TX` (`48`) | 42 | 61,203 | 21.1% | 6,218 | 122 | 9 |
| `Baton Rouge` | `LA` (`22`) | 42 | 36,390 | 27.2% | 5,609 | 10 | 9 |
| `Cincinnati` | `OH` (`39`) | 42 | 55,101 | 23.1% | 6,596 | 152 | 9 |

## 4. Census tract housing tenure & rent burden (`acs_tract_housing_tenure`)

This table comes from the **US Census Bureau American Community Survey (ACS) 5-Year** estimates (`B25070` Gross Rent as a Percentage of Household Income and `B19013` Median Household Income). Housing researchers distinguish two thresholds:

- **Cost-burdened (`30–49%` of income on rent):** `renter_hh_rent_30_to_49_pct_count`
- **Severely cost-burdened (`50%+` of income on rent):** `renter_hh_rent_50_pct_plus_count` — households one car repair, medical bill, or shift cut away from a nonpayment filing.

Notice the five `audit_*` demographic share columns (`audit_black_share`, `audit_hispanic_share`, `audit_asian_share`, `audit_indigenous_share`, `audit_white_share`). Keep them in the table, **never** feed them into your risk score or LLM triage prompt, and use them in Section 11 for your post-output Equity Audit.

In [ ]:
with step("load acs_tract_housing_tenure"):
    raw_tenure = read_starter_parquet("acs_tract_housing_tenure.parquet")
    tenure = raw_tenure[raw_tenure["metro_area"] == METRO].copy()
    tenure["census_tract_geoid"] = safe_str(tenure["census_tract_geoid"], width=11)
    tenure["county_fips"] = safe_str(tenure["county_fips"], width=5)

    sev_share = tenure["renter_hh_rent_50_pct_plus_count"] / tenure["renter_occupied_units"].clip(lower=1)
    spread(tenure["median_gross_rent"], "median_gross_rent ($)")
    spread(sev_share, "severe_rent_burden_share")

    load_table(
        tenure,
        "acs_tract_housing_tenure",
        geom_col="tract_geom",
        description=(
            f"US Census ACS 5-Year renter tenure, rent burden (30-49% and 50%+), poverty, "
            f"vehicle access, and post-output audit_* shares for {METRO} ({STATE_ABBR})."
        ),
    )

## 5. Quarterly court eviction filings (`court_eviction_filings`) & the 4× fan-out trap

County civil courts rarely publish standardized, nationwide, real-time tract-level feeds under an open license. Rather than pretend otherwise, `court_eviction_filings` is explicitly labeled **`[Modeled]`** in `data_provenance`: quarterly tract counts (`2025-Q3`, `2025-Q4`, `2026-Q1`, `2026-Q2`) calibrated to county court filing baselines and ACS structural rent pressure.

**Watch the grain:** there are **four rows per tract** (one per quarter). Join `court_eviction_filings` directly to `municipal_311_housing_complaints` on `census_tract_geoid` without pre-aggregating in a CTE, and every 311 complaint gets counted **four times**.

In [ ]:
with step("load court_eviction_filings"):
    raw_filings = read_starter_parquet("court_eviction_filings.parquet")
    filings = raw_filings[raw_filings["metro_area"] == METRO].copy()
    filings["census_tract_geoid"] = safe_str(filings["census_tract_geoid"], width=11)

    q_counts = filings.groupby("reporting_quarter")[
        ["nonpayment_filings_count", "holdover_or_no_fault_filings_count", "default_judgments_count"]
    ].sum()
    print("Quarterly modeled totals across metro:")
    display(q_counts)

    load_table(
        filings,
        "court_eviction_filings",
        description=(
            f"[Modeled] Quarterly tract-level civil eviction filings (2025-Q3..2026-Q2) "
            f"calibrated to county court baselines and ACS rent burden for {METRO}."
        ),
    )

    # Prove the 4x fan-out trap in one comparison so nobody ships inflated counts on stage.
    naive_join_rows = len(filings.merge(_m311[["request_id", "census_tract_geoid"]], on="census_tract_geoid"))
    print(f"Actual 311 rows in {METRO}                     : {len(_m311):,}")
    print(f"Rows after unaggregated join to 4 quarters     : {naive_join_rows:,}  <-- 4x fan-out!")

## 6. Municipal 311 housing complaints (`municipal_311_housing_complaints`): verifying citizen PII redaction

`block_address` in this file has already been coarsened to the `100 Block` level (`1400 Block ...`), citizen identifiers inside `complaint_narrative` have been pre-scrubbed on disk (`[REDACTED NAME]`, `[REDACTED PLATE]`, `[REDACTED UNIT]`, `[REDACTED STREET NUMBER]`), and `9` synthetic `[Planted Case]` decoys (`DECOY-*`) sit alongside `165` real municipal 311 records per metro.

**Never trust free-text fields from municipal open-data portals without scanning them first.** Raw municipal 311 logs routinely carry self-reported caller names, property manager names, license plates, apartment numbers, and exact street house numbers. Sending unredacted citizen identifiers to an embedding endpoint or displaying them in a demo violates our privacy floor, so we re-apply `scrub_311_narrative()` idempotently before loading to BigQuery (and keep the same redaction guardrails if your team brings additional open-data 311 records).

In [ ]:
_RE_NAME = re.compile(
    r"(?i)\b(my\s+name\s+is\s+|property\s+manager,?\s+)"
    r"[A-Z][A-Za-z'\-]+(?:\s+[A-Z][A-Za-z'\-]+){0,2}"
)
_RE_PLATE = re.compile(r"(?i)(license\s+plate\s*#?\s*)[A-Z0-9\-]+")
_RE_UNIT = re.compile(
    r"(?i)\b(dwelling\s+unit|apt\.?|apartment|room|rm\.?|suite|ste\.?)\s*[:#]?\s*"
    r"\d+[A-Za-z]?(?:\s*(?:,|and)\s*\d+[A-Za-z]?)*"
)
_RE_EXACT_ADDR = re.compile(
    r"(?i)\b(\d{2,5})(?:\s*[/\-]\s*\d{2,5})?\s+(?!Block\b)"
    r"([A-Za-z0-9.\-]+(?:\s+[A-Za-z0-9.\-]+){0,2}\s+"
    r"(?:St|Street|Ave|Avenue|Blvd|Boulevard|Rd|Road|Dr|Drive|Ln|Lane|Ct|Court|Way|Pl|Place)\b)"
)
_RE_SVC_ADDR = re.compile(r"(?i)(service\s+address:\s*)\d{2,5}\b")


def scrub_311_narrative(text):
    if pd.isna(text):
        return text
    cleaned = str(text).replace("\ufffd", "'")
    cleaned = _RE_NAME.sub(r"\1[REDACTED NAME]", cleaned)
    cleaned = _RE_PLATE.sub(r"\1[REDACTED PLATE]", cleaned)
    cleaned = _RE_UNIT.sub(r"\1 [REDACTED UNIT]", cleaned)
    cleaned = _RE_EXACT_ADDR.sub(r"[REDACTED STREET NUMBER] \2", cleaned)
    cleaned = _RE_SVC_ADDR.sub(r"\1[REDACTED STREET NUMBER]", cleaned)
    return cleaned


with step("clean and load municipal_311_housing_complaints"):
    raw_311 = read_starter_parquet("municipal_311_housing_complaints.parquet")
    complaints = raw_311[raw_311["metro_area"] == METRO].copy()
    complaints["census_tract_geoid"] = safe_str(complaints["census_tract_geoid"], width=11)

    before_text = complaints["complaint_narrative"].fillna("").tolist()
    n_unredacted = sum(
        1 for t in before_text
        if _RE_NAME.search(t) or _RE_PLATE.search(t) or _RE_UNIT.search(t)
        or _RE_EXACT_ADDR.search(t) or _RE_SVC_ADDR.search(t)
    )
    n_redacted_tokens = sum(1 for t in before_text if "[REDACTED " in t)
    print(
        f"  PII hygiene check in {METRO}: {n_redacted_tokens} pre-redacted narratives verified, "
        f"{n_unredacted} residual unredacted matches (idempotent scrub applied)"
    )

    complaints["complaint_narrative"] = complaints["complaint_narrative"].map(scrub_311_narrative)
    complaints["created_date"] = pd.to_datetime(complaints["created_date"]).dt.date
    complaints["closed_date"] = pd.to_datetime(complaints["closed_date"]).dt.date

    print("  311 breakdown (property_type x open/closed):")
    display(
        complaints.assign(is_open=complaints["closed_date"].isna())
        .groupby(["property_type", "is_open", "inspection_status"])
        .size()
        .rename("cases")
        .reset_index()
    )

    load_table(
        complaints,
        "municipal_311_housing_complaints",
        geom_col="property_geom",
        description=(
            f"Block-coarsened PII-scrubbed municipal 311 housing complaints (165 real + "
            f"9 [Planted Case] decoys) for {METRO} ({STATE_ABBR})."
        ),
    )

## 7. Emergency rental relief bulletins (`rental_relief_program_bulletins`)

Each metro ships with **four** structured program bulletins (`[Curated Benchmark]`):

- **Two general arrears/stabilization funds** (`requires_open_habitability_case = FALSE`) gated on Area Median Income (AMI), rental arrears documentation, or court summons urgency.
- **Two habitability-contingent funds** (`requires_open_habitability_case = TRUE`)—such as code-enforcement relocation revolving funds or habitability defense grants—that **only** unlock when a residential household also has an active, unclosed habitability complaint.

In [ ]:
with step("load rental_relief_program_bulletins"):
    raw_bulletins = read_starter_parquet("rental_relief_program_bulletins.parquet")
    bulletins = raw_bulletins[raw_bulletins["metro_area"] == METRO].copy()
    bulletins["effective_date"] = pd.to_datetime(bulletins["effective_date"]).dt.date

    display(
        bulletins[
            [
                "bulletin_id",
                "program_name",
                "requires_open_habitability_case",
                "residential_tenants_only",
                "funding_stream",
            ]
        ]
    )

    load_table(
        bulletins,
        "rental_relief_program_bulletins",
        description=(
            f"[Curated Benchmark] Emergency rental relief program bulletins and eligibility "
            f"rules for {METRO} ({STATE_ABBR})."
        ),
    )

## 8. Municipal & state housing statutes (`municipal_housing_ordinance_statutes`)

Three statutory summaries per metro (`[Public Legal Reference]`) ground your coordinator's timeline citations—covering landlord duty-to-repair deadlines, local code-enforcement remedies, and anti-retaliation protections after a tenant reports a code violation. Remind users in your UI that statutory summaries support **internal coordinator triage**, not formal legal representation.

In [ ]:
with step("load municipal_housing_ordinance_statutes"):
    raw_statutes = read_starter_parquet("municipal_housing_ordinance_statutes.parquet")
    statutes = raw_statutes[raw_statutes["metro_area"] == METRO].copy()

    display(statutes[["statute_section_id", "jurisdiction", "code_citation", "ordinance_title"]])

    load_table(
        statutes,
        "municipal_housing_ordinance_statutes",
        description=(
            f"[Public Legal Reference] State and municipal habitability, repair-deadline, "
            f"and anti-retaliation statutory summaries for {METRO} ({STATE_ABBR})."
        ),
    )

## 9. Inventory what landed in BigQuery

Verify row counts and storage metadata directly from BigQuery's `__TABLES__` catalog before writing multi-table analytical queries.

In [ ]:
with step("inventory loaded BigQuery tables"):
    inv = bq.query(f"""
        SELECT
            table_id,
            row_count,
            ROUND(size_bytes / 1024.0, 1) AS size_kb
        FROM `{PROJECT}.{DATASET}.__TABLES__`
        ORDER BY table_id
    """).to_dataframe()
    display(inv)

## 10. Compounding displacement hotspots — the fan-out-safe join

Now let's answer the coordinator's first morning question: **which census tracts combine severe rent burden (`50%+`), heavy annual eviction filing volume, and active residential habitability complaints?**

Notice how `annual_filings` and `open_res_311` are each aggregated to one row per `census_tract_geoid` *before* joining to `acs_tract_housing_tenure`, completely avoiding the 4× quarterly fan-out from Section 5.

In [ ]:
with step("compounding hotspot payoff query"):
    hotspots = bq.query(f"""
        WITH annual_filings AS (
            SELECT
                census_tract_geoid,
                SUM(nonpayment_filings_count + holdover_or_no_fault_filings_count) AS annual_filings_total,
                SUM(default_judgments_count) AS annual_defaults_total,
                ROUND(AVG(median_claimed_arrears_usd), 0) AS avg_claimed_arrears_usd
            FROM `{PROJECT}.{DATASET}.court_eviction_filings`
            GROUP BY census_tract_geoid
        ),
        open_res_311 AS (
            SELECT
                census_tract_geoid,
                COUNT(*) AS open_residential_311_count,
                COUNTIF(request_id LIKE 'DECOY-%') AS planted_open_res_count
            FROM `{PROJECT}.{DATASET}.municipal_311_housing_complaints`
            WHERE property_type = 'Residential'
              AND closed_date IS NULL
            GROUP BY census_tract_geoid
        )
        SELECT
            a.census_tract_geoid,
            a.county_name,
            a.renter_occupied_units,
            ROUND(SAFE_DIVIDE(a.renter_hh_rent_50_pct_plus_count, a.renter_occupied_units) * 100, 1)
                AS severe_rent_burden_pct,
            a.median_gross_rent,
            COALESCE(f.annual_filings_total, 0) AS annual_filings_total,
            COALESCE(f.annual_defaults_total, 0) AS annual_defaults_total,
            COALESCE(f.avg_claimed_arrears_usd, 0) AS avg_claimed_arrears_usd,
            COALESCE(c.open_residential_311_count, 0) AS open_residential_311_count
        FROM `{PROJECT}.{DATASET}.acs_tract_housing_tenure` AS a
        LEFT JOIN annual_filings AS f USING (census_tract_geoid)
        LEFT JOIN open_res_311 AS c USING (census_tract_geoid)
        ORDER BY open_residential_311_count DESC, annual_filings_total DESC
        LIMIT 10
    """).to_dataframe()
    display(hotspots)

## 11. Deliberate exclusions, 311 under-reporting bias, and the Equity Audit rule

Two responsible-AI rules govern every submission on this challenge:

1. **`audit_*` columns are for post-output auditing, never model inputs.** Do **not** pass `audit_black_share`, `audit_hispanic_share`, `audit_asian_share`, `audit_indigenous_share`, or `audit_white_share` into a ranking formula, regression, or LLM prompt to score which neighborhood or household gets help. Compute your triage recommendations strictly from housing need, eviction pressure, habitability severity, and program rules—then join `audit_*` *afterward* to measure who your recommendations reach and who they leave out.
2. **311 complaint volume is not ground-truth habitability prevalence.** Tenants facing immigration fear, month-to-month lease retaliation, or limited English access often **do not call 311**, even when living with broken heat or sewage leaks. A coordinator tool that allocates outreach *only* where 311 calls are high will systematically miss "silent distress" tracts: neighborhoods with high severe rent burden and high eviction filings but near-zero 311 complaints. Let's surface those tracts right now.

In [ ]:
with step("silent-distress & equity audit preview"):
    silent_distress = bq.query(f"""
        WITH annual_filings AS (
            SELECT
                census_tract_geoid,
                SUM(nonpayment_filings_count + holdover_or_no_fault_filings_count) AS annual_filings_total
            FROM `{PROJECT}.{DATASET}.court_eviction_filings`
            GROUP BY census_tract_geoid
        ),
        open_res_311 AS (
            SELECT
                census_tract_geoid,
                COUNT(*) AS open_residential_311_count
            FROM `{PROJECT}.{DATASET}.municipal_311_housing_complaints`
            WHERE property_type = 'Residential' AND closed_date IS NULL
            GROUP BY census_tract_geoid
        )
        SELECT
            a.census_tract_geoid,
            ROUND(SAFE_DIVIDE(a.renter_hh_rent_50_pct_plus_count, a.renter_occupied_units) * 100, 1)
                AS severe_rent_burden_pct,
            COALESCE(f.annual_filings_total, 0) AS annual_filings_total,
            COALESCE(c.open_residential_311_count, 0) AS open_residential_311_count,
            ROUND(a.no_vehicle_rate * 100, 1) AS no_vehicle_pct,
            ROUND(a.audit_black_share * 100, 1) AS audit_black_pct,
            ROUND(a.audit_hispanic_share * 100, 1) AS audit_hispanic_pct
        FROM `{PROJECT}.{DATASET}.acs_tract_housing_tenure` AS a
        LEFT JOIN annual_filings AS f USING (census_tract_geoid)
        LEFT JOIN open_res_311 AS c USING (census_tract_geoid)
        WHERE COALESCE(c.open_residential_311_count, 0) <= 1
        ORDER BY annual_filings_total DESC, severe_rent_burden_pct DESC
        LIMIT 8
    """).to_dataframe()
    print("High eviction-pressure tracts with <=1 open 311 complaint ('silent distress' candidates):")
    display(silent_distress)

## 12. Validation — graded against the loaded BigQuery tables

Every check below queries BigQuery directly—never the in-memory DataFrames—so what gets verified is the exact state your agent and `c_housing_90_publish_snapshot.ipynb` will read. Any `FAIL` raises `RuntimeError` loudly.

In [ ]:
with step("validate loaded BigQuery tables"):
    S = f"{PROJECT}.{DATASET}"

    t_stats = list(bq.query(f"""
        SELECT
            COUNT(*) AS n,
            COUNT(DISTINCT census_tract_geoid) AS n_distinct,
            COUNTIF(LENGTH(census_tract_geoid) != 11) AS bad_geoid_len,
            COUNTIF(SUBSTR(census_tract_geoid, 1, 2) != '{STATE_FIPS}') AS bad_fips_prefix,
            COUNTIF(metro_area != '{METRO}') AS wrong_metro,
            COUNTIF(renter_occupied_units <= 0) AS nonpos_renters,
            COUNTIF(renter_hh_rent_50_pct_plus_count > renter_occupied_units) AS impossible_burden,
            COUNTIF(
                audit_black_share < 0 OR audit_black_share > 1
                OR audit_hispanic_share < 0 OR audit_hispanic_share > 1
                OR audit_asian_share < 0 OR audit_asian_share > 1
                OR audit_indigenous_share < 0 OR audit_indigenous_share > 1
                OR audit_white_share < 0 OR audit_white_share > 1
            ) AS bad_audit_bounds
        FROM `{S}.acs_tract_housing_tenure`
    """).result())[0]

    check("tenure: >= 40 tracts loaded", t_stats.n >= 40, f"{t_stats.n} tracts")
    check("tenure: unique census_tract_geoid", t_stats.n == t_stats.n_distinct, f"{t_stats.n_distinct} distinct")
    check("tenure: all tract IDs are 11 chars (leading zero intact)", t_stats.bad_geoid_len == 0, f"{t_stats.bad_geoid_len} bad")
    check("tenure: all tracts match metro state FIPS", t_stats.bad_fips_prefix == 0, f"expected prefix {STATE_FIPS}")
    check("tenure: all rows belong to selected METRO", t_stats.wrong_metro == 0, f"{t_stats.wrong_metro} wrong")
    check("tenure: positive renter_occupied_units", t_stats.nonpos_renters == 0, f"{t_stats.nonpos_renters} <= 0")
    check("tenure: 50%+ rent-burden count <= renter units", t_stats.impossible_burden == 0, f"{t_stats.impossible_burden} bad")
    check("tenure: audit_* shares within [0, 1]", t_stats.bad_audit_bounds == 0, f"{t_stats.bad_audit_bounds} out of bounds")

    f_stats = list(bq.query(f"""
        SELECT
            COUNT(*) AS n,
            COUNT(DISTINCT filing_record_id) AS n_distinct,
            COUNT(DISTINCT reporting_quarter) AS n_quarters,
            COUNTIF(metro_area != '{METRO}') AS wrong_metro,
            COUNTIF(nonpayment_filings_count < 0 OR holdover_or_no_fault_filings_count < 0) AS neg_counts,
            COUNTIF(census_tract_geoid NOT IN (
                SELECT census_tract_geoid FROM `{S}.acs_tract_housing_tenure`
            )) AS orphan_tracts
        FROM `{S}.court_eviction_filings`
    """).result())[0]

    check("filings: exact 4 quarters per tract", f_stats.n == 4 * t_stats.n and f_stats.n_quarters == 4, f"{f_stats.n} rows across {f_stats.n_quarters} quarters")
    check("filings: unique filing_record_id", f_stats.n == f_stats.n_distinct, f"{f_stats.n_distinct} distinct")
    check("filings: all rows match METRO", f_stats.wrong_metro == 0, f"{f_stats.wrong_metro} wrong")
    check("filings: non-negative filing counts", f_stats.neg_counts == 0, f"{f_stats.neg_counts} negative")
    check("filings: 100% FK match to acs_tract_housing_tenure", f_stats.orphan_tracts == 0, f"{f_stats.orphan_tracts} orphans")

    c_stats = list(bq.query(rf"""
        SELECT
            COUNT(*) AS n,
            COUNT(DISTINCT request_id) AS n_distinct,
            COUNTIF(metro_area != '{METRO}') AS wrong_metro,
            COUNTIF(LENGTH(census_tract_geoid) != 11) AS bad_geoid_len,
            COUNTIF(census_tract_geoid NOT IN (
                SELECT census_tract_geoid FROM `{S}.acs_tract_housing_tenure`
            )) AS orphan_tracts,
            COUNTIF(request_id LIKE 'DECOY-%') AS n_decoys,
            COUNTIF(request_id LIKE 'DECOY-%' AND property_type = 'Commercial' AND closed_date IS NULL) AS decoy_comm_open,
            COUNTIF(request_id LIKE 'DECOY-%' AND property_type = 'Residential' AND inspection_status = 'Closed - Abated') AS decoy_res_abated,
            COUNTIF(request_id LIKE 'DECOY-%' AND property_type = 'Residential' AND inspection_status = 'Closed - No Violation') AS decoy_res_noviol,
            COUNTIF(property_type = 'Residential' AND closed_date IS NULL AND request_id NOT LIKE 'DECOY-%') AS open_real_res,
            COUNTIF(REGEXP_CONTAINS(complaint_narrative, r'(?i)\bmy\s+name\s+is\s+[A-Z][a-z]+')) AS pii_names_left,
            COUNTIF(REGEXP_CONTAINS(complaint_narrative, r'(?i)license\s+plate\s*#?\s*[A-Z0-9]{4,}')) AS pii_plates_left,
            COUNTIF(REGEXP_CONTAINS(complaint_narrative, r'(?i)\b(dwelling\s+unit|apt\.?|apartment|room)\s*[:#]?\s*\d+')) AS pii_units_left
        FROM `{S}.municipal_311_housing_complaints`
    """).result())[0]

    check("311: expected 174 rows (165 real + 9 decoys)", c_stats.n == 174, f"{c_stats.n} rows")
    check("311: unique request_id", c_stats.n == c_stats.n_distinct, f"{c_stats.n_distinct} distinct")
    check("311: all rows match METRO", c_stats.wrong_metro == 0, f"{c_stats.wrong_metro} wrong")
    check("311: all tract IDs are 11 chars", c_stats.bad_geoid_len == 0, f"{c_stats.bad_geoid_len} bad")
    check("311: 100% FK match to acs_tract_housing_tenure", c_stats.orphan_tracts == 0, f"{c_stats.orphan_tracts} orphans")
    check("311: all 9 [Planted Case] decoys present (3+3+3)", c_stats.n_decoys == 9 and c_stats.decoy_comm_open == 3 and c_stats.decoy_res_abated == 3 and c_stats.decoy_res_noviol == 3, f"decoys={c_stats.n_decoys} ({c_stats.decoy_comm_open}/{c_stats.decoy_res_abated}/{c_stats.decoy_res_noviol})")
    check("311: at least 5 open real residential complaints", c_stats.open_real_res >= 5, f"{c_stats.open_real_res} open residential cases")
    check("311 PII scrub: zero self-disclosed caller names remain", c_stats.pii_names_left == 0, f"{c_stats.pii_names_left} remaining")
    check("311 PII scrub: zero license plates remain", c_stats.pii_plates_left == 0, f"{c_stats.pii_plates_left} remaining")
    check("311 PII scrub: zero apartment/room numbers remain", c_stats.pii_units_left == 0, f"{c_stats.pii_units_left} remaining")

    if c_stats.open_real_res < 25:
        note(
            "311 source mix: high share of closed historical 311 requests in this metro",
            165 - c_stats.open_real_res,
            165,
            detail=f"{c_stats.open_real_res} open of 165 real 311 rows in {METRO} (closed cases test your closed_date IS NULL filter)",
            limit=1.0,
        )

    b_stats = list(bq.query(f"""
        SELECT
            COUNT(*) AS n_bul,
            COUNTIF(requires_open_habitability_case) AS n_hab_req,
            COUNTIF(NOT requires_open_habitability_case) AS n_general,
            (SELECT COUNT(*) FROM `{S}.municipal_housing_ordinance_statutes` WHERE metro_area = '{METRO}') AS n_stat
        FROM `{S}.rental_relief_program_bulletins`
        WHERE metro_area = '{METRO}'
    """).result())[0]

    check("bulletins: 4 metro programs (2 general + 2 habitability-contingent)", b_stats.n_bul == 4 and b_stats.n_hab_req == 2 and b_stats.n_general == 2, f"total={b_stats.n_bul} (hab={b_stats.n_hab_req}, gen={b_stats.n_general})")
    check("statutes: 3 metro housing ordinance summaries loaded", b_stats.n_stat == 3, f"{b_stats.n_stat} statutes")

    geo_cols = bq.query(f"""
        SELECT table_name, column_name, data_type
        FROM `{S}.INFORMATION_SCHEMA.COLUMNS`
        WHERE (table_name = 'acs_tract_housing_tenure' AND column_name = 'tract_geom')
           OR (table_name = 'municipal_311_housing_complaints' AND column_name = 'property_geom')
    """).to_dataframe()
    n_geog = int((geo_cols["data_type"] == "GEOGRAPHY").sum()) if len(geo_cols) else 0
    check("spatial: tract_geom and property_geom landed as native GEOGRAPHY", n_geog == 2, f"{n_geog}/2 GEOGRAPHY columns")

    failed = [c for c in CHECKS if c["verdict"] == "FAIL"]
    warned = [c for c in CHECKS if c["verdict"] == "WARN"]
    print(f"\nValidation summary: {len(CHECKS)} checks — {len(CHECKS) - len(failed) - len(warned)} PASS, {len(warned)} WARN, {len(failed)} FAIL")
    if failed:
        raise RuntimeError("Validation failed: " + "; ".join(f"{c['check']} ({c['detail']})" for c in failed))

## 13. Framing your differentiator — BigQuery Vector Search (`AI.EMBED` + `VECTOR_SEARCH`)

This is where the starter notebook stops and your team's engineering begins.

### Step A — Create the Vertex AI `CLOUD_RESOURCE` connection (once per project)

In Cloud Shell (replacing `PROJECT_ID` dynamically via `gcloud config get-value project`):

```bash
PROJECT_ID="$(gcloud config get-value project)"
bq mk --connection --location=US --project_id="${PROJECT_ID}" \
  --connection_type=CLOUD_RESOURCE a4i-vertex

SA="$(bq show --format=json --connection "${PROJECT_ID}.us.a4i-vertex" \
  | python3 -c 'import sys,json; print(json.load(sys.stdin)["cloudResource"]["serviceAccountId"])')"

gcloud projects add-iam-policy-binding "${PROJECT_ID}" \
  --member="serviceAccount:${SA}" \
  --role="roles/aiplatform.user"
```

*(Wait ~60 seconds after the IAM binding for Vertex AI permission propagation.)*

### Step B — Embed 311 narratives & run post-filtered `VECTOR_SEARCH`

Replace `a4i_housing` below if you changed `DATASET`:

```sql
-- 1. Materialize embeddings once (costs pennies and avoids re-embedding on every agent turn)
CREATE OR REPLACE TABLE `a4i_housing.complaints_embedded` AS
SELECT
  request_id, state_code, metro_area, census_tract_geoid, block_address,
  property_type, raw_complaint_type, inspection_status, closed_date, complaint_narrative,
  embedded.result AS embedding
FROM (
  SELECT *,
    AI.EMBED(
      complaint_narrative,
      connection_id => 'us.a4i-vertex',
      endpoint => 'gemini-embedding-2'
    ) AS embedded
  FROM `a4i_housing.municipal_311_housing_complaints`
)
WHERE embedded.status = '';

-- 2. Exact brute-force semantic search + relational Residential & Open filter
SELECT
  base.request_id,
  base.block_address,
  base.census_tract_geoid,
  base.property_type,
  base.inspection_status,
  base.closed_date,
  ROUND(distance, 4) AS cosine_distance,
  base.complaint_narrative
FROM VECTOR_SEARCH(
  TABLE `a4i_housing.complaints_embedded`,
  'embedding',
  (
    SELECT AI.EMBED(
      'no heat in apartment radiators stone cold and ceiling leaking onto electrical box',
      connection_id => 'us.a4i-vertex',
      endpoint => 'gemini-embedding-2'
    ).result AS query_vec
  ),
  query_column_to_search => 'query_vec',
  top_k => 15,
  options => '{"use_brute_force":true}'
)
WHERE base.property_type = 'Residential'
  AND base.closed_date IS NULL
ORDER BY distance ASC;
```

### Five traps that will cost you an hour if you miss them

1. **`AI.EMBED` returns a `STRUCT<result ARRAY<FLOAT64>, status STRING>`, not a bare array.** Pass `.result` into `VECTOR_SEARCH` and filter `WHERE embedded.status = ''` so transient empty arrays never poison the table.
2. **Do not build a `VECTOR INDEX` on 174 rows.** BigQuery refuses or wastes minutes building an IVF/TreeAH index below ~5,000 rows. Pass `options => '{"use_brute_force":true}'` for exact sub-second cosine search.
3. **`top_k` runs *before* your outer `WHERE` clause.** If 6 of the closest 10 semantic matches are `Commercial` or `Closed - Abated` decoys (`DECOY-*`), `top_k => 5` can leave you with almost nothing after filtering. Request `top_k => 15` or `20` (or pre-filter a residential-open view) and apply `WHERE base.property_type = 'Residential' AND base.closed_date IS NULL`.
4. **Embed `rental_relief_program_bulletins` and `municipal_housing_ordinance_statutes` too—or join them deliberately.** A strong coordinator agent matches a household's crisis both to semantically similar 311 habitability evidence *and* to the specific relief bulletin (`requires_open_habitability_case`) and statutory repair deadline (`code_citation`) that apply.
5. **Do not confuse tract risk with individual household facts.** `acs_tract_housing_tenure` and `court_eviction_filings` describe **neighborhood context**, not whether a specific tenant owes back rent. Your agent should explain both layers separately.

### Four questions judges will ask from your 3-minute video & deck

1. **Decoy resilience:** Show what happens when someone queries for `"no heat"` or `"black mold"`. Does your agent reject `DECOY-*` (`Commercial`, `Closed - Abated`, `Closed - No Violation`) and explain *why*?
2. **Compounding triage logic:** How do you weigh an acute life-safety habitability hazard in a moderate-eviction tract against a nonpayment arrears surge in a "silent distress" tract where tenants are afraid to call 311?
3. **Provenance honesty:** Does your UI/deck clearly distinguish real ACS & municipal 311 rows from `[Modeled]` quarterly eviction filings and `[Planted Case]` decoys?
4. **Post-output Equity Audit:** After generating your top priority tracts and program allocations, what did joining `audit_*` and `no_vehicle_rate` reveal—and how did you adjust outreach (e.g., mobile intake vans for high-eviction, low-311 tracts) without using race as a scoring input?

## 14. Diagnostic summary

If anything fails or looks off, paste the output block below to a coach—it captures project, metro, table schemas, check verdicts, and step timings in one place.

In [ ]:
print("=" * 72)
print("A4I HOUSING STABILITY — DIAGNOSTIC SUMMARY")
print("=" * 72)
print(f"project     : {PROJECT}")
print(f"dataset     : {PROJECT}.{DATASET} ({LOCATION})")
print(f"metro       : {METRO}, {STATE_ABBR} (state FIPS {STATE_FIPS})")
print("-" * 72)
print("TABLES")
for tbl_item in bq.list_tables(f"{PROJECT}.{DATASET}"):
    tbl_obj = bq.get_table(tbl_item)
    cols_preview = ", ".join(f"{f.name}:{f.field_type}" for f in tbl_obj.schema[:6])
    print(f"  {tbl_obj.table_id:<38} {tbl_obj.num_rows:>6,} rows  [{cols_preview}, ...]")
print("-" * 72)
print("CHECKS")
for ck in CHECKS:
    print(f"  {ck['verdict']:<5} {ck['check']}: {ck['detail']}")
print("-" * 72)
print("TIMING")
for s_name, secs in STEPS.items():
    print(f"  {s_name:<44} {secs:>6.1f}s")
print(f"  {'TOTAL WALL CLOCK':<44} {time.time() - NOTEBOOK_START:>6.1f}s")
print("=" * 72)